# Train a TinyStories transformer

This follows the embedding → causal attention → blocks → next-token training structure of `transformer.ipynb`. Choose either the local TinyStories tokenizer or the pretrained GPT-2 tokenizer with `TOKENIZER_SOURCE`. Batches are tokenized as they are read, so the whole dataset does not have to be tokenized into memory. The model is trained **from scratch**; only the tokenizer is loaded.

By default this trains for one full epoch over the TinyStories training split and validates on the full validation split. For a quick check, set `MAX_TRAIN_STEPS` and `MAX_VAL_STEPS` below to small integers before running the notebook. This is not the checkpoint/tokenizer format used by `transformer.ipynb`.

In [6]:
from itertools import islice
import time

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

MAX_LEN = 400
BATCH_SIZE = 32
TOKENIZER_SOURCE = 'local'  # 'local' or 'gpt2'
D_MODEL = 23
LAYERS = 2
EPOCHS = 1
MAX_TRAIN_STEPS = None  # None = entire train split each epoch
MAX_VAL_STEPS = None    # None = entire validation split

device = (torch.accelerator.current_accelerator()
          if torch.accelerator.is_available() else torch.device('cpu'))



In [7]:
if TOKENIZER_SOURCE == 'local':
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file('tinystories_tokenizer.json')
    TEXT_VOCAB_SIZE = tokenizer.get_vocab_size()
    BOT_ID = TEXT_VOCAB_SIZE
    EOS_ID = TEXT_VOCAB_SIZE + 1  # local end-of-text ID
    PAD_ID = TEXT_VOCAB_SIZE + 2
    MODEL_VOCAB_SIZE = TEXT_VOCAB_SIZE + 3

    def encode_texts(texts):
        return [item.ids[:MAX_LEN - 1] for item in tokenizer.encode_batch(texts, add_special_tokens=False)]
elif TOKENIZER_SOURCE == 'gpt2':
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    EOS_ID = tokenizer.eos_token_id
    PAD_ID = len(tokenizer)  # EOS remains a valid target
    MODEL_VOCAB_SIZE = len(tokenizer) + 1

    def encode_texts(texts):
        return tokenizer(texts, add_special_tokens=False, truncation=True,
                         max_length=MAX_LEN - 1)['input_ids']
else:
    raise ValueError(f'Unknown TOKENIZER_SOURCE: {TOKENIZER_SOURCE!r}')

In [8]:
print(f'{device=}, {MODEL_VOCAB_SIZE=}, {PAD_ID=}')

device=device(type='xpu'), MODEL_VOCAB_SIZE=2051, PAD_ID=2050


In [9]:
dataset = load_dataset('roneneldan/TinyStories')
train_stories = dataset['train'].select_columns('text')
val_stories = dataset['validation'].select_columns('text')
print(f'Train: {len(train_stories):,} stories; validation: {len(val_stories):,} stories')

def collate_fn(examples):
    encoded = encode_texts([example['text'] for example in examples])
    sequences = [ids + [EOS_ID] for ids in encoded]
    width = max(len(ids) for ids in sequences)
    batch = torch.full((len(sequences), width), PAD_ID, dtype=torch.long)
    for row, ids in enumerate(sequences):
        batch[row, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    return batch

train_loader = DataLoader(train_stories, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, pin_memory=device.type != 'cpu')
val_loader = DataLoader(val_stories, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, pin_memory=device.type != 'cpu')
batch = next(iter(train_loader))
print(f'Batch: {tuple(batch.shape)}, pinned: {batch.is_pinned()}')
assert batch.is_pinned() == (device.type != 'cpu')
assert batch.max() < MODEL_VOCAB_SIZE

Train: 2,119,719 stories; validation: 21,990 stories
Batch: (32, 400), pinned: True


## Causal transformer (same general shape as the original notebook)

In [10]:
def attention_math(q, k, v, mask, scale):
    scores = (q @ k.transpose(-2, -1)) * scale
    scores = scores.masked_fill(~mask[:q.shape[1], :q.shape[1]], float('-inf'))
    return torch.softmax(scores, dim=-1) @ v

class Attention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(MAX_LEN, MAX_LEN, dtype=torch.bool)))
        self.scale = d ** -0.5

    def forward(self, x):
        return attention_math(self.q(x), self.k(x), self.v(x), self.mask, self.scale)

class Embed(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.pos = nn.Embedding(MAX_LEN, d)
        self.tok = nn.Embedding(MODEL_VOCAB_SIZE, d)
        self.dropout = nn.Dropout(0.1)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1], device=ids.device)
        return self.dropout(self.tok(ids) + self.pos(positions))

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attention = Attention(d)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Transformer(nn.Module):
    def __init__(self, d, layers):
        super().__init__()
        self.embed = Embed(d)
        self.blocks = nn.Sequential(*(Block(d) for _ in range(layers)))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, MODEL_VOCAB_SIZE, bias=False)

    def forward(self, ids):
        return self.head(self.norm(self.blocks(self.embed(ids))))

model = Transformer(D_MODEL, LAYERS).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 115,644


## Training with direct batch copies

Batches are copied directly on the training thread. Use `benchmark_transformer_xpu.py` to compare isolated copy latency with compute and full training steps.

In [11]:
def batch_metrics(pred, actual):
    with torch.no_grad():
        mask = actual != PAD_ID
        correct = ((pred.argmax(dim=-1) == actual) & mask).sum().item()
        tokens = mask.sum().item()
    return correct, tokens

def run_epoch(model, loader, opt=None, max_steps=None):
    training = opt is not None
    model.train(training)
    total_loss = total_correct = total_tokens = 0
    count = min(len(loader), max_steps) if max_steps is not None else len(loader)
    start = time.perf_counter()
    with torch.set_grad_enabled(training):
        for host_batch in tqdm(islice(loader, max_steps), total=count):
            batch = host_batch.to(device, non_blocking=True)
            if training:
                opt.zero_grad()
            pred = model(batch[:, :-1])

            loss = F.cross_entropy(pred.reshape(-1, MODEL_VOCAB_SIZE),
                           batch[:, 1:].reshape(-1), ignore_index=PAD_ID)
            if training:
                loss.backward()
                opt.step()

            correct, tokens = batch_metrics(pred, batch[:, 1:])
            
            total_loss += loss.item() * tokens
            total_correct += correct
            total_tokens += tokens
    return {'loss': total_loss / total_tokens,
            'accuracy': total_correct / total_tokens, 'tokens': total_tokens,
            'seconds': time.perf_counter() - start}

In [12]:
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('train:', run_epoch(model, train_loader, opt, MAX_TRAIN_STEPS))
    print('validation:', run_epoch(model, val_loader, max_steps=MAX_VAL_STEPS))

Epoch 1/1


  0%|          | 0/66242 [00:00<?, ?it/s]

KeyboardInterrupt: 